# LeetCode 3229 — Minimum Operations to Make Array Equal to Target bằng Python

- Link: https://leetcode.com/problems/minimum-operations-to-make-array-equal-to-target/
- Độ khó: **Hard**
- Chủ đề: Quy về **mảng chênh lệch** `d = target − nums`; mỗi thao tác cộng/trừ 1 trên một đoạn ⇒ đếm số lần một "tầng" mới bắt đầu (giống LC 1526 — Minimum Number of Increments on Subarrays), xử lý **riêng phần dương và phần âm**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumOperations(self, nums: List[int], target: List[int]) -> int:
```

## 1. Mục tiêu học

1. Chỉ hiệu `d[i] = target[i] − nums[i]` quan trọng; cần biến `d` về toàn 0 bằng các thao tác "+1/−1 trên một đoạn".
2. Phân tách `d = P − N` với `P = max(d, 0)`, `N = max(−d, 0)`: dùng thao tác `+1` cho phần dương, `−1` cho phần âm; trộn chiều không bao giờ có lợi.
3. Với một "dãy núi" không âm `h`, số đoạn `+1` tối thiểu để dựng `h` từ 0 là `Σ max(0, h[i] − h[i−1])` (với `h[−1] = 0`) — số lần một tầng mới phải **mở**.
4. Đáp án `= Σ max(0, P[i] − P[i−1]) + Σ max(0, N[i] − N[i−1])`, `O(n)`.
5. Kiểm chứng bằng BFS trên trạng thái nhỏ.

## 2. Đề bài & ví dụ

Cho hai mảng dương `nums`, `target` cùng độ dài. Mỗi thao tác chọn một **đoạn con** của `nums` và cộng 1 vào mọi phần tử **hoặc** trừ 1 khỏi mọi phần tử. Tìm số thao tác ít nhất để `nums = target`.

**Ví dụ 1:** `nums = [3,5,1,2]`, `target = [4,6,2,4]` → `2`. **Ví dụ 2:** `nums = [1,3,2]`, `target = [2,1,4]` → `5`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`; `1 ≤ nums[i], target[i] ≤ 10^8`.

In [1]:
from typing import List
from collections import deque

examples = [([3, 5, 1, 2], [4, 6, 2, 4], 2), ([1, 3, 2], [2, 1, 4], 5)]
for a, b, exp in examples:
    print(a, b, '-> kỳ vọng', exp, '| d =', [y - x for x, y in zip(a, b)])

[3, 5, 1, 2] [4, 6, 2, 4] -> kỳ vọng 2 | d = [1, 1, 1, 2]
[1, 3, 2] [2, 1, 4] -> kỳ vọng 5 | d = [1, -2, 2]


## 3. Kiến thức nền

### 3.1. LC 1526: dựng "dãy núi" bằng các đoạn `+1`

Với `h ≥ 0`, mỗi khi `h[i] > h[i−1]`, phải mở thêm `h[i] − h[i−1]` đoạn mới bắt đầu tại `i`; khi giảm, đóng bớt đoạn (miễn phí). Tổng số đoạn = `Σ max(0, h[i] − h[i−1])`.

In [2]:
def build_cost(h):
    prev = total = 0
    for x in h:
        if x > prev:
            total += x - prev
        prev = x
    return total

print(build_cost([1, 1, 2]), build_cost([3, 1, 3]), build_cost([0, 2, 0, 2]))

2 5 4


### 3.2. Tách dương / âm

Một thao tác `+1` trên đoạn chứa cả vị trí `d > 0` và `d < 0` làm vị trí âm "xa" 0 hơn ⇒ phải bù bằng `−1` — không bao giờ tốt hơn tách riêng. Nên đáp án là tổng chi phí dựng `P` và dựng `N` độc lập.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `d = [1, −2, 2]` ⇒ `P = [1, 0, 2]`, `N = [0, 2, 0]`.

- `P`: mở 1 tại `i = 0`, đóng tại `i = 1`, mở 2 tại `i = 2` ⇒ `3`.
- `N`: mở 2 tại `i = 1` ⇒ `2`.
- Tổng `5` ✓.

Ví dụ 1: `d = [1, 1, 1, 2]` ⇒ `P` mở 1 rồi thêm 1 ⇒ `2` ✓.

## 5. Cách 1 — Brute Force: BFS trên vector chênh lệch

In [3]:
def brute(nums, target):
    d0 = tuple(y - x for x, y in zip(nums, target)); n = len(d0)
    if not any(d0):
        return 0
    seen = {d0}; q = deque([(d0, 0)])
    bound = max(map(abs, d0)) + 1
    while q:
        d, k = q.popleft()
        for l in range(n):
            for r in range(l, n):
                for delta in (1, -1):
                    nd = list(d)
                    for t in range(l, r + 1):
                        nd[t] -= delta
                    nd = tuple(nd)
                    if max(map(abs, nd)) > bound or nd in seen:
                        continue
                    if not any(nd):
                        return k + 1
                    seen.add(nd); q.append((nd, k + 1))
    return -1

for a, b, exp in examples:
    print(brute(a, b), '| kỳ vọng', exp)

2 | kỳ vọng 2
5 | kỳ vọng 5


**Độ phức tạp:** mũ theo `n` và độ lớn — chỉ cho `n ≤ 4`, `|d| ≤ 3`. (Giới hạn `|d| ≤ max|d₀| + 1` không làm mất nghiệm tối ưu — kiểm chứng thực nghiệm khớp với công thức.)

## 6. Ý tưởng thuật toán / bước đột phá

```
ans = 0; prev = 0
for i: cur = target[i] − nums[i]
    if cur > 0:  ans += max(0, cur − max(prev, 0))
    if cur < 0:  ans += max(0, −cur − max(−prev, 0))
    prev = cur
```

Một lần duyệt, `O(n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Chia để trị "trừ đáy" (đệ quy như LC 1526)

Trên mỗi đoạn cùng dấu, trừ đi giá trị tuyệt đối nhỏ nhất (một số thao tác trên cả đoạn), rồi đệ quy trên các đoạn còn khác 0. `O(n²)` xấu nhất.

In [4]:
def divide_conquer(nums, target):
    d = [y - x for x, y in zip(nums, target)]
    def solve(arr):
        # arr: các giá trị cùng dấu (hoặc 0)
        total = 0; i = 0; n = len(arr)
        while i < n:
            if arr[i] == 0:
                i += 1; continue
            j = i
            while j < n and arr[j] != 0 and (arr[j] > 0) == (arr[i] > 0):
                j += 1
            seg = arr[i:j]
            m = min(abs(x) for x in seg)
            sgn = 1 if seg[0] > 0 else -1
            total += m + solve([x - sgn * m for x in seg])
            i = j
        return total
    return solve(d)

for a, b, exp in examples:
    print(divide_conquer(a, b), '| kỳ vọng', exp)

2 | kỳ vọng 2
5 | kỳ vọng 5


### 7.2. Cách 3 — Công thức tầng dương/âm (lời giải chính)

In [5]:
def layers(nums, target):
    ans = prev = 0
    for x, y in zip(nums, target):
        cur = y - x
        if cur > 0:
            ans += max(0, cur - max(prev, 0))
        elif cur < 0:
            ans += max(0, -cur - max(-prev, 0))
        prev = cur
    return ans

for a, b, exp in examples:
    print(layers(a, b), '| kỳ vọng', exp)

2 | kỳ vọng 2
5 | kỳ vọng 5


### 7.3. Cách 4 — Tách hẳn hai mảng `P`, `N` rồi gọi LC 1526 hai lần

In [6]:
def split_then_1526(nums, target):
    d = [y - x for x, y in zip(nums, target)]
    P = [max(v, 0) for v in d]
    N = [max(-v, 0) for v in d]
    return build_cost(P) + build_cost(N)

for a, b, exp in examples:
    print(split_then_1526(a, b), '| kỳ vọng', exp)

2 | kỳ vọng 2
5 | kỳ vọng 5


### 7.4. Cách 5 — Một dòng: tổng các bước **lên** của `d` có lính canh 0 hai đầu

Bất ngờ nhưng đúng: `ans = Σ max(0, d[i] − d[i−1])` với `d[−1] = d[n] = 0` (tương đương `½ · Σ |d[i] − d[i−1]|`). Lý do: viết `Δd = ΔP − ΔN` (giá đỡ của `P` và `N` rời nhau nên mỗi bước, phần tăng của `d` gồm phần tăng của `P` cộng phần giảm của `N`). Tổng lại: bước lên của `d` = (bước lên của `P`) + (bước xuống của `N`) = (bước lên của `P`) + (bước lên của `N`) vì `N` bắt đầu và kết thúc ở 0. Một đoạn `−1` được "trả tiền" khi nó **đóng** (d đi lên về 0) thay vì khi mở — tổng không đổi.

In [7]:
def up_steps(nums, target):
    d = [0] + [y - x for x, y in zip(nums, target)] + [0]
    return sum(max(0, d[i] - d[i - 1]) for i in range(1, len(d)))

def half_variation(nums, target):
    d = [0] + [y - x for x, y in zip(nums, target)] + [0]
    return sum(abs(d[i] - d[i - 1]) for i in range(1, len(d))) // 2

for a, b, exp in examples:
    print(up_steps(a, b), half_variation(a, b), '| kỳ vọng', exp)

2 2 | kỳ vọng 2
5 5 | kỳ vọng 5


Hai đẳng thức nhỏ đáng nhớ: với dãy bắt đầu và kết thúc ở 0, **tổng bước lên = tổng bước xuống = ½ tổng biến thiên**. Cách 5 gọn nhất, nhưng cách 3 dễ giải thích "đang mở bao nhiêu đoạn dấu nào" hơn khi phỏng vấn, và nó không cần lính canh cuối.

## 8. TRACE từng bước

Trace cách 3 trên `nums = [1, 1, 1, 1, 1, 1]`, `target = [3, 5, 2, 0, -1 + 1, 4]` (tức `d = [2, 4, 1, −1, −1, 3]`).

In [8]:
def trace(nums, target):
    ans = prev = 0
    for i, (x, y) in enumerate(zip(nums, target)):
        cur = y - x
        add = max(0, cur - max(prev, 0)) if cur > 0 else (max(0, -cur - max(-prev, 0)) if cur < 0 else 0)
        ans += add
        print(f'i = {i}: d = {cur:2d} (trước {prev:2d}) -> mở thêm {add} đoạn {"+1" if cur > 0 else "-1" if cur < 0 else "  "} | tổng {ans}')
        prev = cur
    print('đáp án:', ans, '| chia để trị:', divide_conquer(nums, target))

trace([1, 1, 1, 1, 1, 1], [3, 5, 2, 0, 0, 4])

i = 0: d =  2 (trước  0) -> mở thêm 2 đoạn +1 | tổng 2
i = 1: d =  4 (trước  2) -> mở thêm 2 đoạn +1 | tổng 4
i = 2: d =  1 (trước  4) -> mở thêm 0 đoạn +1 | tổng 4
i = 3: d = -1 (trước  1) -> mở thêm 1 đoạn -1 | tổng 5
i = 4: d = -1 (trước -1) -> mở thêm 0 đoạn -1 | tổng 5
i = 5: d =  3 (trước -1) -> mở thêm 3 đoạn +1 | tổng 8
đáp án: 8 | chia để trị: 8


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def minimumOperations(self, nums: List[int], target: List[int]) -> int:
        ans = 0
        prev = 0
        for x, y in zip(nums, target):
            cur = y - x
            if cur > 0:
                if prev > 0:
                    if cur > prev:
                        ans += cur - prev
                else:
                    ans += cur
            elif cur < 0:
                if prev < 0:
                    if cur < prev:
                        ans += prev - cur
                else:
                    ans -= cur
            prev = cur
        return ans

for a, b, exp in examples:
    got = Solution().minimumOperations(a, b)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
5 OK


## 10. Vì sao thuật toán đúng

1. **Cận dưới:** xét ranh giới giữa `i − 1` và `i` với `P[i] > P[i−1]`. Mỗi thao tác `+1` chứa `i` nhưng không chứa `i − 1` làm tăng `(P[i] − P[i−1])` "tương đối" tối đa 1; thao tác `−1` chỉ làm tình hình xấu hơn với phần dương. Do đó cần `≥ P[i] − P[i−1]` thao tác **bắt đầu** tại `i` (và tương tự cho `N`); các thao tác bắt đầu tại các vị trí khác nhau là khác nhau ⇒ `ans ≥ Σ max(0, ΔP) + Σ max(0, ΔN)`.
2. **Đạt được:** dựng `P` bằng tham lam LC 1526 (mỗi tầng một đoạn tối đại), dựng `N` tương tự bằng `−1`; hai việc độc lập vì `P`, `N` có giá đỡ rời nhau.
3. Hằng số `max(prev, 0)` / `max(−prev, 0)` đúng là `P[i−1]` / `N[i−1]`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS | mũ | mũ |
| 2. Chia để trị | `O(n²)` xấu nhất | `O(n)` |
| 3. Công thức tầng | `O(n)` | `O(1)` |
| 4. Tách + 1526 | `O(n)` | `O(n)` |
| 5. Bước lên / ½ biến thiên | `O(n)` | `O(n)` (hoặc `O(1)` nếu duyệt trực tiếp) |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'divide_conquer': divide_conquer, 'layers': layers, 'split_1526': split_then_1526, 'up_steps': up_steps, 'half_var': half_variation,
         'submit': lambda a, b: Solution().minimumOperations(a, b)}
edge = [([5], [5], 0), ([1], [4], 3), ([4], [1], 3), ([1, 1], [2, 0 + 1], 1), ([2, 2, 2], [3, 1, 3], 3), ([1, 2, 3], [3, 2, 1], 4)]
for a, b, exp in examples + edge:
    assert brute(a, b) == exp, (a, b, brute(a, b))
    for name, f in impls.items():
        assert f(a, b) == exp, (name, a, b)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với BFS

In [11]:
import random
rng = random.Random(3229)
cnt = 0
for _ in range(250):
    n = rng.randint(1, 4)
    nums = [rng.randint(1, 4) for _ in range(n)]
    target = [max(1, x + rng.randint(-3, 3)) for x in nums]
    exp = brute(nums, target)
    for name, f in impls.items():
        assert f(nums, target) == exp, (name, nums, target)
    cnt += 1
print('Khớp BFS trên', cnt, 'cặp mảng ngẫu nhiên.')

Khớp BFS trên 250 cặp mảng ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`, giá trị tới `10^8`

In [12]:
import time
N = 10 ** 5
big = {
    'ngẫu nhiên': ([rng.randint(1, 10 ** 8) for _ in range(N)], [rng.randint(1, 10 ** 8) for _ in range(N)]),
    'cùng dấu tăng dần': (list(range(1, N + 1)), [2 * i for i in range(1, N + 1)]),
    'đổi dấu xen kẽ': ([5] * N, [6 if i % 2 else 4 for i in range(N)]),
}
for name, (a, b) in big.items():
    t = time.time(); x = Solution().minimumOperations(a, b); t1 = time.time() - t
    y = split_then_1526(a, b)
    assert x == y
    print(f"{name:18s}: {x:14d} | nộp {t1:.2f}s")
assert Solution().minimumOperations(list(range(1, N + 1)), [2 * i for i in range(1, N + 1)]) == N
assert Solution().minimumOperations([5] * N, [6 if i % 2 else 4 for i in range(N)]) == N
print('d = 1..N (tăng) -> N đoạn lồng nhau; xen kẽ ±1 -> N thao tác đơn lẻ: đúng.')

ngẫu nhiên        :  2333594180432 | nộp 0.01s
cùng dấu tăng dần :         100000 | nộp 0.01s


đổi dấu xen kẽ    :         100000 | nộp 0.01s
d = 1..N (tăng) -> N đoạn lồng nhau; xen kẽ ±1 -> N thao tác đơn lẻ: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng `Σ |d[i]|` (mỗi đơn vị một thao tác riêng)

In [13]:
def wrong_sum_abs(nums, target):
    return sum(abs(y - x) for x, y in zip(nums, target))            # SAI

print('sai :', wrong_sum_abs(*examples[0][:2]), '| đúng:', Solution().minimumOperations(*examples[0][:2]))

sai : 5 | đúng: 2


**Vì sao sai:** một thao tác tác động **cả đoạn** — `d = [1, 1, 1, 2]` chỉ cần 2 thao tác.

### Lỗi 2 — Dùng `|d|` liên tục qua chỗ đổi dấu

In [14]:
def wrong_abs_layers(nums, target):
    ans = prev = 0
    for x, y in zip(nums, target):
        cur = abs(y - x)
        ans += max(0, cur - prev)                                     # SAI: tầng dương nối với tầng âm
        prev = cur
    return ans

print('sai :', wrong_abs_layers([2, 2], [3, 1]), '| đúng:', Solution().minimumOperations([2, 2], [3, 1]))

sai : 1 | đúng: 2


**Vì sao sai:** `d = [1, −1]`: một đoạn `+1` không thể "tiếp tục" thành đoạn `−1` — cần hai thao tác.

### Lỗi 3 — Lấy `max |d|`

In [15]:
def wrong_max_abs(nums, target):
    return max(abs(y - x) for x, y in zip(nums, target))            # SAI

print('sai :', wrong_max_abs(*examples[1][:2]), '| đúng:', Solution().minimumOperations(*examples[1][:2]))

sai : 2 | đúng: 5


**Vì sao sai:** chỉ đúng khi `d` là một "ngọn núi" đơn cùng dấu; thung lũng giữa hai đỉnh buộc mở tầng lại.

### Lỗi 4 — Dùng "tổng bước lên của d" nhưng quên lính canh 0 ở **cuối**

In [16]:
def wrong_no_tail(nums, target):
    d = [0] + [y - x for x, y in zip(nums, target)]                   # SAI: thiếu số 0 ở cuối
    return sum(max(0, d[i] - d[i - 1]) for i in range(1, len(d)))

print('sai :', wrong_no_tail([3], [1]), '| đúng:', Solution().minimumOperations([3], [1]))
print('sai :', wrong_no_tail([5, 5], [6, 3]), '| đúng:', Solution().minimumOperations([5, 5], [6, 3]))

sai : 0 | đúng: 2
sai : 1 | đúng: 3


**Vì sao sai:** với cách 5 (bước lên của `d`), một đoạn `−1` được "tính tiền" ở bước **đi lên** trở về 0 — nếu `d` kết thúc âm thì bước đó nằm ở lính canh cuối. `d = [−2]` cho 0 thay vì 2.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cộng/trừ 1 trên đoạn con" ⇒ nghĩ theo **mảng hiệu** / số đoạn **mở** tại mỗi vị trí.
- Hai chiều (+/−) ⇒ tách phần dương và phần âm.
- LC 1526, LC 3229, "Minimum Number of Operations to Make Array Continuous"… cùng họ.

## 15. Biến thể của bài

1. **LC 1526:** chỉ có `+1`, xuất phát từ 0.
2. Thao tác cộng/trừ `k` thay vì 1 ⇒ cần `d[i] mod k = 0`, chia `d` cho `k`.
3. Mảng **vòng** (đoạn có thể bọc qua cuối) ⇒ đếm bước lên theo vòng, cộng thêm `min(h)` cho các tầng phủ kín vòng (bài tập 4).
4. Chi phí thao tác tỉ lệ độ dài đoạn ⇒ trở thành `Σ |d[i]|`.

In [17]:
# Biến thể 2: bước k
def step_k(nums, target, k):
    d = [y - x for x, y in zip(nums, target)]
    if any(v % k for v in d):
        return -1
    return layers([0] * len(d), [v // k for v in d])

for _ in range(100):
    n = rng.randint(1, 4); k = rng.randint(1, 3)
    nums = [rng.randint(1, 5) for _ in range(n)]
    target = [x + k * rng.randint(-2, 2) for x in nums]
    if min(target) < 1:
        continue
    assert step_k(nums, target, k) == brute([0] * n, [(y - x) // k for x, y in zip(nums, target)])
print('Biến thể bước k: quy về bài gốc trên d / k — khớp BFS.')

Biến thể bước k: quy về bài gốc trên d / k — khớp BFS.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | BFS | mũ | Oracle |
| 2 | Chia để trị trừ đáy | `O(n²)` | Trực quan |
| 3 | Công thức tầng | `O(n)` | **Nộp bài** |
| 4 | Tách `P`, `N` + 1526 | `O(n)` | Rõ ràng nhất |
| 5 | Bước lên của `d` (½ biến thiên) | `O(n)` | Ngắn nhất |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [1, 1, 1]`, `target = [2, 2, 2]`.

**Bài 2 (tính tay):** `nums = [5, 5, 5, 5]`, `target = [7, 3, 7, 3]`.

**Bài 3 (điền TODO):** trả về luôn **danh sách thao tác** `(l, r, ±1)` đạt tối ưu.

**Bài 4 (tự viết lại):** biến thể 3 — mảng vòng (cho `n` nhỏ, kiểm bằng BFS vòng).

**Bài 5 (mini challenge):** số thao tác ít nhất khi chỉ được dùng `+1` (không có `−1`), cho phép **chọn** thêm một hằng `c` cộng vào toàn bộ `target` (tức `target + c`, `c ≥ 0` sao cho mọi `d ≥ 0`).

In [18]:
def operations_list_todo(nums, target):
    # TODO
    raise NotImplementedError

def circular_todo(nums, target):
    # TODO
    raise NotImplementedError

def shift_then_increment_todo(nums, target):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `d = [1, 1, 1]`. (2) Một đoạn. (3) `1`.

**Bài 2.** (1) `d = [2, −2, 2, −2]`. (2) `P` mở 2 tại `0` và `2`; `N` mở 2 tại `1` và `3`. (3) `8`.

**Bài 3.** (1) Dùng ngăn xếp các đoạn đang mở cho mỗi dấu. (2) Khi `P` giảm, đóng bớt (ghi lại `(l, i−1)`). (3) Cuối mảng đóng hết.

**Bài 4.** (1) Trên vòng, mỗi "tầng" `{h ≥ k}` là hợp các cung. (2) Số cung của tầng = số bước lên (tính vòng, `h[−1]` = phần tử cuối), trừ khi tầng phủ kín vòng — khi đó cần 1 cung tròn đầy. (3) Chi phí `= Σ_vòng max(0, h[i] − h[i−1]) + min(h)` cho mỗi phần `P`, `N`; kiểm bằng BFS với các đoạn bọc vòng.

**Bài 5.** (1) Với `c` cố định: `d' = d + c ≥ 0`, chi phí `build_cost(d')`. (2) `c` càng lớn càng tốn? Mở tầng đầu tăng `c` nhưng các chỗ tăng khác không đổi ⇒ chọn `c` nhỏ nhất `= max(0, −min d)`. (3) Kiểm bằng thử nhiều `c`.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().minimumOperations([1, 1, 1], [2, 2, 2]) == 1
assert Solution().minimumOperations([5, 5, 5, 5], [7, 3, 7, 3]) == 8

def operations_list(nums, target):
    d = [y - x for x, y in zip(nums, target)] + [0]
    ops = []
    for sign in (1, -1):
        h = [max(sign * v, 0) for v in d]
        stack = []                                  # vị trí bắt đầu của các tầng đang mở
        prev = 0
        for i, x in enumerate(h):
            while prev < x:
                stack.append(i); prev += 1
            while prev > x:
                ops.append((stack.pop(), i - 1, sign)); prev -= 1
    return ops

for _ in range(200):
    n = rng.randint(1, 6); a = [rng.randint(1, 6) for _ in range(n)]; b = [rng.randint(1, 6) for _ in range(n)]
    ops = operations_list(a, b)
    cur = list(a)
    for l, r, s in ops:
        for t in range(l, r + 1):
            cur[t] += s
    assert cur == b and len(ops) == Solution().minimumOperations(a, b)
print('Bài 1: 1 | Bài 2: 8 | Bài 3: danh sách thao tác đúng và tối ưu. Ví dụ 2:', operations_list(*examples[1][:2]))

Bài 1: 1 | Bài 2: 8 | Bài 3: danh sách thao tác đúng và tối ưu. Ví dụ 2: [(0, 0, 1), (2, 2, 1), (2, 2, 1), (1, 1, -1), (1, 1, -1)]


In [20]:
# Bài 4, 5
def brute_circular(nums, target):
    d0 = tuple(y - x for x, y in zip(nums, target)); n = len(d0)
    if not any(d0):
        return 0
    arcs = [[(l + t) % n for t in range(L)] for l in range(n) for L in range(1, n + 1)]
    seen = {d0}; q = deque([(d0, 0)]); bound = max(map(abs, d0)) + 1
    while q:
        d, k = q.popleft()
        for arc in arcs:
            for s in (1, -1):
                nd = list(d)
                for t in arc:
                    nd[t] -= s
                nd = tuple(nd)
                if max(map(abs, nd)) > bound or nd in seen:
                    continue
                if not any(nd):
                    return k + 1
                seen.add(nd); q.append((nd, k + 1))

def circular(nums, target):
    # Trên vòng: tầng k của h là {h >= k}; số cung = số bước "lên" vòng ở mức k,
    # riêng khi tầng phủ kín vòng (k <= min h) thì cần đúng 1 cung tròn đầy.
    d = [y - x for x, y in zip(nums, target)]; n = len(d)
    total = 0
    for sign in (1, -1):
        h = [max(sign * v, 0) for v in d]
        total += sum(max(0, h[i] - h[i - 1]) for i in range(n)) + min(h)   # h[-1] là phần tử cuối (vòng)
    return total

def shift_then_increment(nums, target):
    d = [y - x for x, y in zip(nums, target)]
    c = max(0, -min(d))
    return build_cost([v + c for v in d])

for _ in range(120):
    n = rng.randint(1, 4); a = [rng.randint(1, 3) for _ in range(n)]; b = [max(1, x + rng.randint(-2, 2)) for x in a]
    assert circular(a, b) == brute_circular(a, b), (a, b)
    d = [y - x for x, y in zip(a, b)]; c0 = max(0, -min(d))
    assert shift_then_increment(a, b) == min(build_cost([v + c for v in d]) for c in range(c0, c0 + 6))
print('Bài 4 (vòng) khớp BFS vòng; Bài 5: c nhỏ nhất là tối ưu.')

Bài 4 (vòng) khớp BFS vòng; Bài 5: c nhỏ nhất là tối ưu.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ans = prev = 0
for x, y in zip(nums, target):
    cur = y - x
    if cur > 0: ans += max(0, cur - max(prev, 0))
    elif cur < 0: ans += max(0, -cur - max(-prev, 0))
    prev = cur
```

**Checklist:**

- [ ] Làm việc trên `d = target − nums`?
- [ ] Tách phần dương và âm (không nối tầng qua chỗ đổi dấu)?
- [ ] Chỉ cộng khi tầng **tăng**?
- [ ] `prev` khởi tạo 0?
- [ ] Kết quả tới `~10^{13}` (64-bit ở ngôn ngữ khác)?